# MiniProject 3 - CDO Risk, Part 1
**FRE-GY 6103 | September 2026**

Simulation workspace for the simplified CDO analysis: fixed draws, correlated
default times, BIS collateral cash flows, and tranche waterfall allocations.
Statistical analysis and valuation remain for later work.
All monetary amounts are in **USD**, times are in **years**, and rates are decimals.

Run cells in order using the **Python (CDO Risk)** kernel.

### Reading the code
The model uses explicit loops, named intermediate values, and ordinary dictionaries.
`table.loc[case_id, bond_id]` selects by labels; `table.iloc[row, column]` selects by position.
`to_numpy()` extracts the numerical array from a pandas table. Array positions start at 0;
case IDs and ranks start at 1. In arrays, `axis=0` means cases, `axis=1` means bonds
(or quarters in a case-by-quarter table). Bond cash flows have a third axis for quarters.
`@` is matrix multiplication and `.T` is transpose, as in our Cholesky derivation.
`assert` and `np.testing.assert_allclose` are checks that stop execution if results are inconsistent.
The slider code is display plumbing: focus first on the calculation functions above it.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def format_number(value):
    return f'{value:,.6f}'

pd.set_option('display.float_format', format_number)
project_dir = Path.cwd()
if not (project_dir / 'cdo_risk_part1.ipynb').exists():
    project_dir = project_dir / 'MiniProject3'
if not (project_dir / 'cdo_risk_part1.ipynb').exists():
    raise FileNotFoundError('Start this notebook from the MiniProject3 folder.')
data_dir = project_dir / 'data'
data_dir.mkdir(exist_ok=True)
print(f'Python: {sys.version.split()[0]}')
print(f'Interpreter: {sys.executable}')
print(f'NumPy {np.__version__} | pandas {pd.__version__} | SciPy {scipy.__version__}')


Python: 3.13.0
Interpreter: C:\Users\Andre\iCloud Drive_new\iCloudDrive\Desktop\NYU FA26\FRE-GY 6103\MiniProject3\.venv\Scripts\python.exe
NumPy 2.5.3 | pandas 3.0.6 | SciPy 1.18.1


## 1. Assignment inputs
Provided in the assignment description. The five-year term has 20 quarterly payment dates.


In [2]:
SEED = 6103
N_CASES = 1_000
N_BONDS = 10
MATURITY_YEARS = 5
PAYMENTS_PER_YEAR = 4
FACE_VALUE = 10_000_000.0
ANNUAL_COUPON = 0.06
ANNUAL_DEFAULT_PROBABILITY = 0.04
LGD = 0.60
DEFAULT_CORRELATION = 0.20
MARKET_YTM = 0.09
RISK_FREE_RATE = 0.01

tranches = pd.DataFrame({
    'class': ['A', 'B'],
    'initial_notional_usd': [20_000_000.0, 10_000_000.0],
    'annual_coupon': [0.02, 0.04],
}).set_index('class')
case_ids = pd.Index(range(1, N_CASES + 1), name='case_id')
bond_names = []
for bond_number in range(1, N_BONDS + 1):
    bond_names.append(f'bond_{bond_number:02d}')
bond_ids = pd.Index(bond_names, name='bond')
N_PERIODS = MATURITY_YEARS * PAYMENTS_PER_YEAR
payment_times = np.arange(1, N_PERIODS + 1) / PAYMENTS_PER_YEAR

inputs = pd.Series({
    'cases': N_CASES, 'bonds': N_BONDS, 'maturity_years': MATURITY_YEARS,
    'payments_per_year': PAYMENTS_PER_YEAR, 'face_value_per_bond_usd': FACE_VALUE,
    'total_collateral_face_usd': FACE_VALUE * N_BONDS,
    'annual_coupon': ANNUAL_COUPON,
    'annual_default_probability': ANNUAL_DEFAULT_PROBABILITY,
    'loss_given_default': LGD, 'recovery_fraction': 1 - LGD,
    'assignment_correlation': DEFAULT_CORRELATION,
    'market_ytm': MARKET_YTM, 'risk_free_rate': RISK_FREE_RATE,
}, name='value')
display(inputs.to_frame())
display(tranches)
display(pd.DataFrame({'quarter': np.arange(1, N_PERIODS + 1), 'time_years': payment_times}))


,value
cases,"1,000.000000"
bonds,10.000000
maturity_years,5.000000
payments_per_year,4.000000
face_value_per_bond_usd,"10,000,000.000000"
total_collateral_face_usd,"100,000,000.000000"
annual_coupon,0.060000
annual_default_probability,0.040000
loss_given_default,0.600000
recovery_fraction,0.400000


,initial_notional_usd,annual_coupon
class,,
A,"20,000,000.000000",0.020000
B,"10,000,000.000000",0.040000


,quarter,time_years
0,1,0.250000
1,2,0.500000
2,3,0.750000
3,4,1.000000
4,5,1.250000
5,6,1.500000
6,7,1.750000
7,8,2.000000
8,9,2.250000
9,10,2.500000


## 2. Fixed random-number foundation
Generate 1,000 rows of 10 **independent standard-normal draws** using seed 6103.
These are base inputs, not correlated default times or completed simulation results.
The first run saves the draws to `data/base_normal_draws.csv`; later runs reload
that file so parameter changes can use the same underlying randomness.

Keep this file for Part 2. Changing the seed alone does not replace an existing
file. Intentional regeneration requires renaming the saved file first.

**Moment matching (Lecture 4, slide 10):** standardize each bond column across the 1,000 cases using its sample mean and sample standard deviation (`ddof=1`), before Cholesky. The original CSV stays unchanged. This matches marginal sample moments, not the full sample covariance matrix.


In [3]:
draws_path = data_dir / 'base_normal_draws.csv'
if draws_path.exists():
    base_draws = pd.read_csv(draws_path, index_col='case_id', float_precision='round_trip')
    print('Loaded existing fixed draws.')
else:
    rng = np.random.Generator(np.random.PCG64(SEED))
    base_draws = pd.DataFrame(
        rng.standard_normal((N_CASES, N_BONDS)), index=case_ids, columns=bond_ids
    )
    base_draws.to_csv(draws_path, float_format='%.17g')
    print(f'Created fixed draws with PCG64 seed {SEED}.')

assert base_draws.shape == (N_CASES, N_BONDS)
assert base_draws.index.equals(case_ids), 'Expected case IDs 1 through 1000.'
assert base_draws.columns.tolist() == bond_ids.tolist()
assert np.isfinite(base_draws.to_numpy()).all()
display(base_draws.head())


# Moment-match one bond column at a time; keep the raw draws unchanged.
matched_draws = base_draws.copy()
moment_rows = []
for bond_id in bond_ids:
    raw_column = base_draws[bond_id]
    column_mean = raw_column.mean()
    column_std = raw_column.std(ddof=1)  # Sample standard deviation: divide by n - 1.
    if column_std <= 0:
        raise ValueError("Moment matching requires a nonconstant column.")

    centered_column = raw_column - column_mean
    matched_draws[bond_id] = centered_column / column_std
    moment_rows.append({
        "bond": bond_id,
        "raw_mean": column_mean,
        "raw_std": column_std,
        "matched_mean": matched_draws[bond_id].mean(),
        "matched_std": matched_draws[bond_id].std(ddof=1),
    })

np.testing.assert_allclose(matched_draws.mean(), 0, atol=1e-12)
np.testing.assert_allclose(matched_draws.std(ddof=1), 1, atol=1e-12)
print("Moment matching passed: column means 0 and sample standard deviations 1.")
moment_summary = pd.DataFrame(moment_rows).set_index("bond")
display(moment_summary)


Loaded existing fixed draws.


,bond_01,bond_02,bond_03,bond_04,bond_05,bond_06,bond_07,bond_08,bond_09,bond_10
case_id,,,,,,,,,,
1,-0.389205,1.157821,0.094404,0.347190,1.110191,0.658477,1.046767,-0.473348,-0.652418,-0.111504
2,-0.743672,1.212835,-0.521107,0.865185,0.493481,0.540806,0.114611,0.804709,0.653957,0.219143
3,1.092946,-1.038571,0.389650,-0.491659,-0.703602,-0.180830,-0.790995,0.122441,-0.219314,-0.470948
4,-0.434196,1.915500,-0.939071,-0.405338,-0.050326,1.311731,-0.881236,-1.606150,-0.910297,-1.065262
5,-1.859987,0.750618,0.222459,-0.783419,-1.526318,-0.785312,-0.834635,-0.079704,-0.832377,0.285846


Moment matching passed: column means 0 and sample standard deviations 1.


,raw_mean,raw_std,matched_mean,matched_std
bond,,,,
bond_01,0.017697,1.005140,0.000000,1.000000
bond_02,-0.006030,1.014729,0.000000,1.000000
bond_03,0.033039,1.001197,0.000000,1.000000
bond_04,-0.067234,0.990737,-0.000000,1.000000
bond_05,-0.017215,1.019728,-0.000000,1.000000
bond_06,-0.037462,1.013273,-0.000000,1.000000
bond_07,-0.012345,1.003197,-0.000000,1.000000
bond_08,0.030573,0.998166,0.000000,1.000000
bond_09,-0.013935,0.979222,0.000000,1.000000


## 3. Inspect the fixed draws
This selector uses original case IDs and shows raw and moment-matched draws. After default times are calculated, the collateral and waterfall selectors use ranks: rank 1 has the most defaults, rank 1000 the fewest. Both ranks and original IDs are displayed. Ties retain original case order; ranking is by default count, not cash-flow loss.

In [4]:
def show_case(case_id=1):
    display(pd.DataFrame({"raw_normal": base_draws.loc[case_id],
                          "matched_normal": matched_draws.loc[case_id]}))

case_viewer = widgets.interactive(
    show_case,
    case_id=widgets.IntSlider(value=1, min=1, max=N_CASES, step=1,
                             description='Original ID:', continuous_update=False),
)
display(case_viewer)


interactive(children=(IntSlider(value=1, continuous_update=False, description='Original ID:', max=1000, min=1)…

In [5]:
# Target covariance/correlation matrix for the standard normal variables.
# Diagonal = variance 1; off-diagonal = pairwise correlation 0.20.
R = np.full((N_BONDS, N_BONDS), DEFAULT_CORRELATION, dtype=float)
np.fill_diagonal(R, 1.0)

# NumPy returns the lower-triangular factor: L @ L.T = R.
L = np.linalg.cholesky(R)

# Check the defining identity, allowing for floating-point rounding.
assert np.allclose(L @ L.T, R), "Cholesky reconstruction failed."
print(f"R shape: {R.shape}; L shape: {L.shape}")
print(f"Maximum reconstruction error: {np.max(np.abs(L @ L.T - R)):.2e}")

print("Target matrix R:")
display(pd.DataFrame(R, index=bond_ids, columns=bond_ids))
print("Lower-triangular Cholesky factor L:")
normal_labels = []
for number in range(1, N_BONDS + 1):
    normal_labels.append(f"Z_{number}")
display(pd.DataFrame(L, index=bond_ids, columns=normal_labels))


R shape: (10, 10); L shape: (10, 10)
Maximum reconstruction error: 1.11e-16
Target matrix R:


bond,bond_01,bond_02,bond_03,bond_04,bond_05,bond_06,bond_07,bond_08,bond_09,bond_10
bond,,,,,,,,,,
bond_01,1.000000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000
bond_02,0.200000,1.000000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000
bond_03,0.200000,0.200000,1.000000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000
bond_04,0.200000,0.200000,0.200000,1.000000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000
bond_05,0.200000,0.200000,0.200000,0.200000,1.000000,0.200000,0.200000,0.200000,0.200000,0.200000
bond_06,0.200000,0.200000,0.200000,0.200000,0.200000,1.000000,0.200000,0.200000,0.200000,0.200000
bond_07,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,1.000000,0.200000,0.200000,0.200000
bond_08,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,1.000000,0.200000,0.200000
bond_09,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,1.000000,0.200000


Lower-triangular Cholesky factor L:


,Z_1,Z_2,Z_3,Z_4,Z_5,Z_6,Z_7,Z_8,Z_9,Z_10
bond,,,,,,,,,,
bond_01,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
bond_02,0.200000,0.979796,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
bond_03,0.200000,0.163299,0.966092,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
bond_04,0.200000,0.163299,0.138013,0.956183,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
bond_05,0.200000,0.163299,0.138013,0.119523,0.948683,0.000000,0.000000,0.000000,0.000000,0.000000
bond_06,0.200000,0.163299,0.138013,0.119523,0.105409,0.942809,0.000000,0.000000,0.000000,0.000000
bond_07,0.200000,0.163299,0.138013,0.119523,0.105409,0.094281,0.938083,0.000000,0.000000,0.000000
bond_08,0.200000,0.163299,0.138013,0.119523,0.105409,0.094281,0.085280,0.934199,0.000000,0.000000
bond_09,0.200000,0.163299,0.138013,0.119523,0.105409,0.094281,0.085280,0.077850,0.930949,0.000000


In [6]:
# Cholesky: rows are cases, so Y = Z @ L.T.
normal_values = matched_draws.to_numpy()
correlated_values = normal_values @ L.T
correlated_draws = pd.DataFrame(
    correlated_values, index=base_draws.index, columns=base_draws.columns
)
assert correlated_draws.shape == base_draws.shape
assert np.isfinite(correlated_values).all()

def bond_pair_correlations(frame):
    """Collect the correlation of each distinct pair of bond columns."""
    correlation_table = frame.corr()
    pair_values = []
    number_of_bonds = len(frame.columns)
    for first_bond in range(number_of_bonds):
        for second_bond in range(first_bond + 1, number_of_bonds):
            pair_values.append(correlation_table.iloc[first_bond, second_bond])
    return np.array(pair_values)

sample_cov = correlated_draws.cov()
sample_corr = correlated_draws.corr()
cov_error = sample_cov.to_numpy() - R
pair_correlations = bond_pair_correlations(correlated_draws)
sample_variances = np.diag(sample_cov)

# The sample covariance transforms just like the theoretical covariance.
matched_covariance = matched_draws.cov().to_numpy()
expected_sample_covariance = L @ matched_covariance @ L.T
np.testing.assert_allclose(sample_cov, expected_sample_covariance)

print(f"Correlated draws shape: {correlated_draws.shape}")
print(f"Largest covariance deviation from R: {np.abs(cov_error).max():.4f}")
print(f"Sample variance range: {sample_variances.min():.4f} to {sample_variances.max():.4f}")
print(f"Mean pairwise sample correlation: {pair_correlations.mean():.4f} (target {DEFAULT_CORRELATION:.2f})")
print(f"Pairwise correlation range: {pair_correlations.min():.4f} to {pair_correlations.max():.4f}")
print("Moment matching does not force the full sample covariance to equal R.")
display(sample_cov.round(4))
display(sample_corr.round(4))


Correlated draws shape: (1000, 10)
Largest covariance deviation from R: 0.0863
Sample variance range: 0.9592 to 1.0600
Mean pairwise sample correlation: 0.2134 (target 0.20)
Pairwise correlation range: 0.1486 to 0.2785
Moment matching does not force the full sample covariance to equal R.


,bond_01,bond_02,bond_03,bond_04,bond_05,bond_06,bond_07,bond_08,bond_09,bond_10
bond_01,1.000000,0.192800,0.197300,0.205400,0.221300,0.215800,0.222400,0.223900,0.196900,0.184300
bond_02,0.192800,0.997100,0.201300,0.205900,0.223400,0.243700,0.162400,0.286300,0.150200,0.215100
bond_03,0.197300,0.201300,1.000000,0.234700,0.218500,0.222300,0.234000,0.245500,0.206900,0.273100
bond_04,0.205400,0.205900,0.234700,1.013600,0.233600,0.238000,0.239100,0.257800,0.159700,0.196300
bond_05,0.221300,0.223400,0.218500,0.233600,1.022900,0.215300,0.215700,0.227100,0.160900,0.254100
bond_06,0.215800,0.243700,0.222300,0.238000,0.215300,1.026300,0.218500,0.284000,0.147400,0.215200
bond_07,0.222400,0.162400,0.234000,0.239100,0.215700,0.218500,1.012400,0.241300,0.187900,0.238800
bond_08,0.223900,0.286300,0.245500,0.257800,0.227100,0.284000,0.241300,1.060000,0.211300,0.232400
bond_09,0.196900,0.150200,0.206900,0.159700,0.160900,0.147400,0.187900,0.211300,0.959200,0.165700
bond_10,0.184300,0.215100,0.273100,0.196300,0.254100,0.215200,0.238800,0.232400,0.165700,1.019900


,bond_01,bond_02,bond_03,bond_04,bond_05,bond_06,bond_07,bond_08,bond_09,bond_10
bond_01,1.000000,0.193000,0.197300,0.204000,0.218800,0.213000,0.221000,0.217500,0.201000,0.182500
bond_02,0.193000,1.000000,0.201600,0.204900,0.221200,0.240900,0.161600,0.278500,0.153500,0.213300
bond_03,0.197300,0.201600,1.000000,0.233200,0.216100,0.219400,0.232600,0.238500,0.211300,0.270400
bond_04,0.204000,0.204900,0.233200,1.000000,0.229400,0.233300,0.236100,0.248700,0.162000,0.193100
bond_05,0.218800,0.221200,0.216100,0.229400,1.000000,0.210100,0.212000,0.218100,0.162400,0.248800
bond_06,0.213000,0.240900,0.219400,0.233300,0.210100,1.000000,0.214300,0.272300,0.148600,0.210400
bond_07,0.221000,0.161600,0.232600,0.236100,0.212000,0.214300,1.000000,0.233000,0.190700,0.235000
bond_08,0.217500,0.278500,0.238500,0.248700,0.218100,0.272300,0.233000,1.000000,0.209500,0.223500
bond_09,0.201000,0.153500,0.211300,0.162000,0.162400,0.148600,0.190700,0.209500,1.000000,0.167600
bond_10,0.182500,0.213300,0.270400,0.193100,0.248800,0.210400,0.235000,0.223500,0.167600,1.000000


## 4. Gaussian copula and default times

For each case and bond, transform the correlated standard normal draw into a percentile:

$$U_{i,j}=\Phi(Y_{i,j}).$$

In the ideal Gaussian model, each marginal is uniform on $(0,1)$; dependence across bonds is retained through the Gaussian copula. Sample moment matching makes these marginal distribution statements approximate at finite sample size. Then use the exponential inverse CDF:

$$T_{i,j}=\frac{\ln(1-U_{i,j})}{\ln(1-p)},\qquad \lambda=-\ln(1-p),\quad p=0.04.$$

Times are in years. A bond defaults during the project horizon when $T\le5$; times beyond five years remain in the data rather than being capped at maturity.

**Model convention:** 0.20 is the latent normal correlation used in the copula. Pearson correlations of uniforms and default times generally differ from it. The assignment’s wording about default-date correlation should be confirmed before finalizing this convention.

**Numerics:** `np.log1p(-u)` evaluates $\ln(1-u)$ accurately. We protect against floating-point values rounded to 0 or 1 and report any affected draws.

**Checks:** compare simulated default frequencies with $1-(1-p)^t$. Moment matching couples rows through shared sample moments. The displayed `naive_SE_scale` is only an independent-case reference scale, not a validated Monte Carlo standard error for this adjusted sample. Rigorous error estimation would require independent batches, each moment-matched separately.

References: Lecture 4, slides 17–20; Lecture 5, slide 19. The lecture’s continuous-time formula corresponds to an exponential distribution (a whole-period count would be geometric).

In [7]:
# 1. Gaussian copula: standard normal CDF gives uniform marginals.
uniform_values = stats.norm.cdf(correlated_draws.to_numpy())
clipped_count = np.count_nonzero((uniform_values <= 0) | (uniform_values >= 1))
# Keep values inside (0, 1) so logarithms stay finite.
smallest_uniform = np.nextafter(0.0, 1.0)
largest_uniform = np.nextafter(1.0, 0.0)
uniform_values = np.clip(uniform_values, smallest_uniform, largest_uniform)
correlated_uniforms = pd.DataFrame(
    uniform_values, index=correlated_draws.index, columns=correlated_draws.columns
)

# 2. Exponential inverse CDF: default times in years.
p = ANNUAL_DEFAULT_PROBABILITY
assert 0 < p < 1
hazard_rate = -np.log1p(-p)
# log1p(-u) computes log(1-u) accurately, including very small u.
default_time_values = -np.log1p(-uniform_values) / hazard_rate
default_times = pd.DataFrame(
    default_time_values,
    index=correlated_draws.index, columns=correlated_draws.columns
)
defaults_by_maturity = default_times <= MATURITY_YEARS

# Structural checks and a round trip through the exponential CDF.
assert default_times.shape == (N_CASES, N_BONDS)
assert np.isfinite(default_times.to_numpy()).all()
assert (default_times.to_numpy() > 0).all()
assert np.allclose(-np.expm1(-hazard_rate * default_times.to_numpy()),
                   correlated_uniforms.to_numpy())
print(f"Hazard rate: {hazard_rate:.6f} per year")
print(f"Uniform draws adjusted at floating-point endpoints: {clipped_count}")

# Count -> dictionary -> sort by the counts (the Top K pattern).
default_count_by_case = {}
for case_id in case_ids:
    count = 0
    for bond_id in bond_ids:
        if defaults_by_maturity.loc[case_id, bond_id]:
            count += 1
    default_count_by_case[case_id] = count

# sorted returns the original IDs, ordered by their dictionary values.
# It is stable: equal counts keep the original case order.
ranked_case_ids = sorted(default_count_by_case,
                         key=default_count_by_case.get, reverse=True)
ranking_rows = []
for case_id in ranked_case_ids:
    ranking_rows.append({"original_case_id": case_id,
                         "default_count": default_count_by_case[case_id]})
case_ranking = pd.DataFrame(ranking_rows)
case_ranking.index = pd.Index(range(1, N_CASES + 1), name="rank")
default_counts = pd.Series(default_count_by_case)

# Reorder every related table using the SAME list of original IDs.
correlated_draws = correlated_draws.loc[ranked_case_ids]
correlated_uniforms = correlated_uniforms.loc[ranked_case_ids]
default_times = default_times.loc[ranked_case_ids]
defaults_by_maturity = defaults_by_maturity.loc[ranked_case_ids]
assert case_ranking["default_count"].is_monotonic_decreasing
assert case_ranking["original_case_id"].is_unique
print("Cases ranked by defaults within maturity (ties keep original order):")
display(case_ranking.head(10))

# Inspect the highest-default case.
selected_case = int(case_ranking.loc[1, "original_case_id"])
print(f"Case {selected_case}: normal -> uniform -> default time")
display(pd.DataFrame({
    "normal_draw": correlated_draws.loc[selected_case],
    "uniform_percentile": correlated_uniforms.loc[selected_case],
    "default_time_years": default_times.loc[selected_case],
    "defaults_within_5_years": defaults_by_maturity.loc[selected_case],
}))

# Compare marginal default frequencies with theory.
# Each row contributes a default fraction; moment matching couples rows.
checks = []
for horizon in [1.0, float(MATURITY_YEARS)]:
    case_default_fraction = (default_times <= horizon).mean(axis=1)
    simulated = case_default_fraction.mean()
    theoretical = -np.expm1(-hazard_rate * horizon)
    naive_se = case_default_fraction.std(ddof=1) / np.sqrt(N_CASES)
    difference = simulated - theoretical
    if naive_se > 0:
        difference_in_naive_se = difference / naive_se
    else:
        difference_in_naive_se = np.nan
    checks.append({
        "horizon_years": horizon,
        "theoretical_probability": theoretical,
        "simulated_probability": simulated,
        "difference": simulated - theoretical,
        "naive_SE_scale": naive_se,
        "difference_in_naive_SE": difference_in_naive_se,
    })
    print(f"{horizon:g}-year defaults: simulated {simulated:.2%}; "
          f"target {theoretical:.2%}; naive SE scale {naive_se:.2%}")
default_frequency_checks = pd.DataFrame(checks).set_index("horizon_years")
display(default_frequency_checks)
print("Frequency checks are approximate; naive SE scales do not account for moment matching.")

# The copula retains dependence, but nonlinear transforms change Pearson correlation.
for label, frame in [("Normals", correlated_draws),
                     ("Uniforms", correlated_uniforms),
                     ("Default times", default_times)]:
    mean_correlation = bond_pair_correlations(frame).mean()
    print(f"{label}: mean pairwise sample Pearson correlation = {mean_correlation:.4f}")

Hazard rate: 0.040822 per year
Uniform draws adjusted at floating-point endpoints: 0
Cases ranked by defaults within maturity (ties keep original order):


,original_case_id,default_count
rank,,
1,618,9
2,760,9
3,140,8
4,321,8
5,562,8
6,850,8
7,899,8
8,65,7
9,74,7


Case 618: normal -> uniform -> default time


,normal_draw,uniform_percentile,default_time_years,defaults_within_5_years
bond_01,-0.947759,0.171626,4.612477,True
bond_02,-1.557871,0.059632,1.506147,True
bond_03,-1.577282,0.057365,1.447173,True
bond_04,-1.099634,0.135746,3.573769,True
bond_05,-1.518756,0.064412,1.630983,True
bond_06,-1.833394,0.033372,0.831453,True
bond_07,-2.277614,0.011375,0.280240,True
bond_08,-0.916060,0.179818,4.855928,True
bond_09,-0.008950,0.496429,16.805434,False
bond_10,-1.681365,0.046346,1.162470,True


1-year defaults: simulated 4.28%; target 4.00%; naive SE scale 0.25%
5-year defaults: simulated 18.56%; target 18.46%; naive SE scale 0.56%


,theoretical_probability,simulated_probability,difference,naive_SE_scale,difference_in_naive_SE
horizon_years,,,,,
1.000000,0.040000,0.042800,0.002800,0.002513,1.114267
5.000000,0.184627,0.185600,0.000973,0.005608,0.173464


Frequency checks are approximate; naive SE scales do not account for moment matching.
Normals: mean pairwise sample Pearson correlation = 0.2134
Uniforms: mean pairwise sample Pearson correlation = 0.2041
Default times: mean pairwise sample Pearson correlation = 0.1863


## 5. BIS collateral cash flows

Lecture 5, slides 5–6, splits every promised payment into a nondefaultable portion $(1-\mathrm{LGD})$ and a defaultable portion $\mathrm{LGD}$. Default eliminates the latter from the next scheduled payment onward.

$$CF_{i,q}=CF_q^{\mathrm{promised}}\left[(1-\mathrm{LGD})+\mathrm{LGD}\,\mathbf{1}_{\{T_i>t_q\}}\right],\qquad t_q=q/4.$$

Each bond promises **$150,000 per quarter**, plus **$10 million principal in quarter 20**. After default, 40% of the scheduled coupons and principal remains: **$60,000 per quarter** and **$4,060,000 in the final quarter**. Recovery follows the original payment schedule; there is no additional lump-sum recovery at default. Payments already made stay unchanged.

**Boundary convention:** default exactly on a payment date reduces that payment; survival requires `default_time > payment_time`. This equality has probability zero in the continuous model, but is tested explicitly.

The cash-flow array has axes **case × bond × quarter**. Summing the bond axis gives a **1,000 × 20** portfolio cash-flow table. Amounts are undiscounted USD; valuation comes later.

In [8]:
# Promised quarterly payments for one bond.
quarter_ids = pd.Index(range(1, N_PERIODS + 1), name="quarter")
quarterly_coupon = FACE_VALUE * ANNUAL_COUPON / PAYMENTS_PER_YEAR
promised_cash_flows = np.full(N_PERIODS, quarterly_coupon, dtype=float)
promised_cash_flows[-1] += FACE_VALUE

def bis_cash_flows(times, lgd=LGD):
    """Return undiscounted USD cash flows with axes: case, bond, quarter.

    times is a 2D array of default times in years; infinity means no default.
    Recovery is the surviving fraction of each original scheduled payment.
    """
    times = np.asarray(times, dtype=float)
    if times.ndim != 2 or np.isnan(times).any() or (times < 0).any():
        raise ValueError("Default times must be a nonnegative 2D array without NaN.")
    if not 0 <= lgd <= 1:
        raise ValueError("LGD must be between 0 and 1.")

    number_of_cases, number_of_bonds = times.shape
    cash_flows = np.zeros((number_of_cases, number_of_bonds, N_PERIODS))

    for case_position in range(number_of_cases):
        for bond_position in range(number_of_bonds):
            default_time = times[case_position, bond_position]
            for quarter_position in range(N_PERIODS):
                payment_time = payment_times[quarter_position]
                promised_payment = promised_cash_flows[quarter_position]

                if default_time > payment_time:
                    # Bond survives this payment date: pay the full amount.
                    actual_payment = promised_payment
                else:
                    # BIS convention: keep the recovery share of this payment.
                    actual_payment = (1 - lgd) * promised_payment

                cash_flows[case_position, bond_position, quarter_position] = actual_payment

    return cash_flows

# Apply the same schedule to every simulated bond and case.
bond_cash_flows = bis_cash_flows(default_times.to_numpy())
# Sum across bonds (axis 1), leaving a case-by-quarter table.
portfolio_values = bond_cash_flows.sum(axis=1)
portfolio_cash_flows = pd.DataFrame(
    portfolio_values, index=default_times.index, columns=quarter_ids
)
assert bond_cash_flows.shape == (N_CASES, N_BONDS, N_PERIODS)
assert np.isfinite(bond_cash_flows).all()
assert (bond_cash_flows >= 0).all()
assert np.all(bond_cash_flows <= promised_cash_flows + 1e-8)
assert np.all(bond_cash_flows >= (1 - LGD) * promised_cash_flows - 1e-8)

# Boundary checks use the current model parameters.
boundary = bis_cash_flows([[0.0, payment_times[0], payment_times[-1], np.inf]])[0]
assert np.allclose(boundary[0], (1 - LGD) * promised_cash_flows)
assert np.allclose(boundary[1], boundary[0])
assert np.allclose(boundary[2, :-1], promised_cash_flows[:-1])
assert np.isclose(boundary[2, -1], (1 - LGD) * promised_cash_flows[-1])
assert np.allclose(boundary[3], promised_cash_flows)
assert np.allclose(bis_cash_flows([[0.0]], lgd=0)[0, 0], promised_cash_flows)
assert np.allclose(bis_cash_flows([[0.0]], lgd=1), 0)

print("BIS cash-flow bounds, payment-date boundaries, and LGD endpoint checks passed.")
print(f"Bond cash-flow array: {bond_cash_flows.shape} (case, bond, quarter)")
print(f"Portfolio cash-flow table: {portfolio_cash_flows.shape} (case, quarter)")

BIS cash-flow bounds, payment-date boundaries, and LGD endpoint checks passed.
Bond cash-flow array: (1000, 10, 20) (case, bond, quarter)
Portfolio cash-flow table: (1000, 20) (case, quarter)


In [9]:
def defaulted_bond_names(case_id):
    """List bonds that default by maturity in this original case."""
    names = []
    for bond_id in bond_ids:
        if defaults_by_maturity.loc[case_id, bond_id]:
            names.append(bond_id)
    return names

def describe_defaulted_bonds(case_id):
    names = defaulted_bond_names(case_id)
    if len(names) == 0:
        return "none"
    return ", ".join(names)

def show_collateral_cash_flows(rank=1, bond_id="bond_01"):
    """Inspect one bond alongside the aggregate cash flows for its case."""
    case_id = int(case_ranking.loc[rank, "original_case_id"])
    print(f"Rank {rank} of {N_CASES} (most defaults first).")
    case_pos = default_times.index.get_loc(case_id)
    bond_pos = default_times.columns.get_loc(bond_id)
    default_time = default_times.loc[case_id, bond_id]
    defaulted_bonds = defaulted_bond_names(case_id)
    default_count = len(defaulted_bonds)
    defaulted_names = describe_defaulted_bonds(case_id)
    print(f"Case {case_id}: {default_count} of {N_BONDS} bonds default by maturity. "
          f"Defaulted bonds: {defaulted_names}.")
    print(f"{bond_id}: default time {default_time:.4f} years; "
          f"defaults by maturity: {default_time <= MATURITY_YEARS}")
    display(pd.DataFrame({
        "time_years": payment_times,
        "promised_bond_USD": promised_cash_flows,
        "actual_bond_USD": bond_cash_flows[case_pos, bond_pos],
        "portfolio_USD": portfolio_cash_flows.loc[case_id].to_numpy(),
    }, index=quarter_ids))


# Start at the highest-default case; show a defaulted bond if available.
initial_case_pos = 0
initial_bond_pos = 0
worst_case_id = int(case_ranking.loc[1, "original_case_id"])
for bond_position in range(N_BONDS):
    bond_id = bond_ids[bond_position]
    if defaults_by_maturity.loc[worst_case_id, bond_id]:
        initial_bond_pos = bond_position
        break
collateral_viewer = widgets.interactive(
    show_collateral_cash_flows,
    rank=widgets.IntSlider(value=1, min=1, max=N_CASES, description="Rank:",
                           continuous_update=False),
    bond_id=widgets.Dropdown(options=default_times.columns.tolist(),
                            value=default_times.columns[initial_bond_pos], description="Bond:"),
)
display(collateral_viewer)


interactive(children=(IntSlider(value=1, continuous_update=False, description='Rank:', max=1000, min=1), Dropd…

## 6. Tranche waterfall

Pay **Class A first, then Class B, then residual equity** each quarter. All amounts are undiscounted USD. No cash or unpaid shortfalls carry forward.

**Principal assumption (pending confirmation):** the assignment names PACs but supplies no amortization schedule. Here both tranches repay all principal in quarter 20, with no early principal payments. This is an explicit bullet-payment assumption, not an inferred PAC schedule. Replace the scheduled payments if a different schedule is supplied.

A has 20 million USD notional at 2% annually; B has 10 million USD at 4%. Each owes **100,000 USD per quarter**. At maturity, total amounts due are **20,100,000 USD for A** and **10,100,000 USD for B**. A's entire scheduled payment (coupon plus principal) takes priority over B's entire scheduled payment.

For available collateral cash $C_q$ and scheduled payments $D_{A,q},D_{B,q}$:

$$A_q=\min(C_q,D_{A,q}),\qquad B_q=\min(C_q-A_q,D_{B,q}),\qquad E_q=C_q-A_q-B_q.$$

Shortfalls are scheduled minus actual payments, recorded for analysis without adding them to later dues. The implementation applies these rules to every case and quarter using explicit loops. The case selector below shows all 20 quarters and totals.

**Model implication:** with 60% LGD, the BIS model preserves 40% of every collateral payment even if all bonds default. Under this principal assumption, this lower bound covers A and B in full. Equity absorbs the collateral losses; zero debt shortfalls here follow from the assumptions, not a guarantee about real CDOs.


In [10]:
# 1. Build the scheduled payments for A and B.
scheduled_tranche_cash_flows = pd.DataFrame(0.0, index=quarter_ids, columns=["A", "B"])
tranche_quarterly_coupons = {}
for tranche_name in ["A", "B"]:
    notional = tranches.loc[tranche_name, "initial_notional_usd"]
    annual_rate = tranches.loc[tranche_name, "annual_coupon"]
    quarterly_payment = notional * annual_rate / PAYMENTS_PER_YEAR
    tranche_quarterly_coupons[tranche_name] = quarterly_payment

    for quarter in quarter_ids:
        amount_due = quarterly_payment
        if quarter == N_PERIODS:
            amount_due += notional  # Assumption: principal due at maturity.
        scheduled_tranche_cash_flows.loc[quarter, tranche_name] = amount_due

def apply_waterfall(collateral, schedule):
    """Return payments by case, quarter, and class (A=0, B=1, equity=2)."""
    collateral = np.asarray(collateral, dtype=float)
    schedule = np.asarray(schedule, dtype=float)
    if collateral.ndim != 2:
        raise ValueError("Collateral must be a case-by-quarter array.")
    number_of_cases, number_of_quarters = collateral.shape
    if schedule.shape != (number_of_quarters, 2):
        raise ValueError("Schedule must have one row per quarter and columns A, B.")
    for values in [collateral, schedule]:
        if not np.isfinite(values).all():
            raise ValueError("Cash flows must be finite.")
        if (values < 0).any():
            raise ValueError("Cash flows cannot be negative.")

    payments = np.zeros((number_of_cases, number_of_quarters, 3))
    for case_position in range(number_of_cases):
        for quarter_position in range(number_of_quarters):
            cash_available = collateral[case_position, quarter_position]
            a_due = schedule[quarter_position, 0]
            b_due = schedule[quarter_position, 1]

            a_paid = min(cash_available, a_due)
            cash_remaining = cash_available - a_paid
            b_paid = min(cash_remaining, b_due)
            equity_paid = cash_remaining - b_paid

            payments[case_position, quarter_position, 0] = a_paid
            payments[case_position, quarter_position, 1] = b_paid
            payments[case_position, quarter_position, 2] = equity_paid
    return payments

# 2. Apply the waterfall to the ranked portfolio cash flows.
schedule_values = scheduled_tranche_cash_flows.to_numpy()
tranche_cash_flows = apply_waterfall(portfolio_cash_flows.to_numpy(), schedule_values)
class_cash_flows = {}
class_names = ["A", "B", "E"]
for class_position in range(len(class_names)):
    class_name = class_names[class_position]
    # Colons select all cases and all quarters for this one class.
    class_values = tranche_cash_flows[:, :, class_position]
    class_cash_flows[class_name] = pd.DataFrame(
        class_values, index=portfolio_cash_flows.index, columns=quarter_ids
    )

# 3. Record shortfalls; do not add them to next quarter's amount due.
tranche_shortfalls = np.zeros((N_CASES, N_PERIODS, 2))
for case_position in range(N_CASES):
    for quarter_position in range(N_PERIODS):
        for class_position in range(2):
            amount_due = schedule_values[quarter_position, class_position]
            amount_paid = tranche_cash_flows[case_position, quarter_position, class_position]
            tranche_shortfalls[case_position, quarter_position, class_position] = amount_due - amount_paid

# 4. Validate payments, conservation and seniority.
assert tranche_cash_flows.shape == (N_CASES, N_PERIODS, 3)
assert np.isfinite(tranche_cash_flows).all()
assert (tranche_cash_flows >= 0).all()
assert (tranche_shortfalls >= -1e-8).all()
total_distributed = tranche_cash_flows.sum(axis=2)
np.testing.assert_allclose(total_distributed, portfolio_cash_flows)
for case_position in range(N_CASES):
    for quarter_position in range(N_PERIODS):
        a_shortfall = tranche_shortfalls[case_position, quarter_position, 0]
        b_shortfall = tranche_shortfalls[case_position, quarter_position, 1]
        b_paid = tranche_cash_flows[case_position, quarter_position, 1]
        equity_paid = tranche_cash_flows[case_position, quarter_position, 2]
        if a_shortfall > 1e-8:
            assert b_paid == 0
        if a_shortfall > 1e-8 or b_shortfall > 1e-8:
            assert equity_paid == 0

minimum_collateral = N_BONDS * (1 - LGD) * promised_cash_flows
# A list containing one schedule represents one all-default case.
minimum_payments = apply_waterfall([minimum_collateral], schedule_values)
debt_fully_covered_at_lower_bound = np.allclose(minimum_payments[0, :, :2], schedule_values)
print(f"Waterfall checks passed for {N_CASES:,} cases x {N_PERIODS} quarters.")
print(f"All-default BIS lower bound covers A and B: {debt_fully_covered_at_lower_bound}")
for class_position in range(2):
    count = 0
    for case_position in range(N_CASES):
        case_shortfalls = tranche_shortfalls[case_position, :, class_position]
        if (case_shortfalls > 1e-8).any():
            count += 1
    print(f"Class {class_names[class_position]}: {count} of {N_CASES} cases have a payment shortfall.")
display(scheduled_tranche_cash_flows)


Waterfall checks passed for 1,000 cases x 20 quarters.
All-default BIS lower bound covers A and B: True
Class A: 0 of 1000 cases have a payment shortfall.
Class B: 0 of 1000 cases have a payment shortfall.


,A,B
quarter,,
1,"100,000.000000","100,000.000000"
2,"100,000.000000","100,000.000000"
3,"100,000.000000","100,000.000000"
4,"100,000.000000","100,000.000000"
5,"100,000.000000","100,000.000000"
6,"100,000.000000","100,000.000000"
7,"100,000.000000","100,000.000000"
8,"100,000.000000","100,000.000000"
9,"100,000.000000","100,000.000000"


In [11]:
def show_waterfall(rank=1):
    """Display C, A, B and E for each quarter of a simulated case."""
    case_id = int(case_ranking.loc[rank, "original_case_id"])
    print(f"Rank {rank} of {N_CASES} (most defaults first).")
    defaulted_names = describe_defaulted_bonds(case_id)
    print(f"Case {case_id}: defaulted bonds by maturity: {defaulted_names}.")
    print("Priority: A -> B -> E. Principal assumption: bullet at maturity.")
    table = pd.DataFrame(index=quarter_ids)
    table["time_years"] = payment_times
    table["C_USD"] = portfolio_cash_flows.loc[case_id].to_numpy()
    for class_name in ["A", "B", "E"]:
        class_table = class_cash_flows[class_name]
        table[class_name + "_USD"] = class_table.loc[case_id].to_numpy()

    for column_name in ["C_USD", "A_USD", "B_USD", "E_USD"]:
        table.loc["Total", column_name] = table[column_name].sum()
    display(table.style.format({"time_years": "{:.2f}", "C_USD": "{:,.2f}",
                                "A_USD": "{:,.2f}", "B_USD": "{:,.2f}",
                                "E_USD": "{:,.2f}"}, na_rep=""))

waterfall_viewer = widgets.interactive(
    show_waterfall,
    rank=widgets.IntSlider(value=1,
                             min=1, max=N_CASES, description="Rank:",
                             continuous_update=False),
)
display(waterfall_viewer)


interactive(children=(IntSlider(value=1, continuous_update=False, description='Rank:', max=1000, min=1), Outpu…

## 7. Statistics and sensitivities - to be implemented
Analyze collateral and tranche cash-flow distributions across the 1,000 cases.
Include means, dispersion, quantiles, shortfalls, and Monte Carlo uncertainty.
Reuse the fixed draws when varying default probability, LGD, and correlation.
Prepare charts and interpretation for a bank-client report after the model is
complete. Final valuation and Part 2 optimization remain for later work.


## Local course references
- [Lecture 4 - Simulation](../Lecture%204%20-%20Simulation.pdf)
- [Lecture 5 - Events, Corporate Bonds and CDOs](../Lecture%205%20-%20Events,%20Corporate%20Bonds%20and%20CDOs.pdf)
- [Lecture 2A - Loans with exponential default risk](../Lecture%202A%20-%20Loans%20with%20exponential%20default%20risk.pdf)

If JupyterLab's project-root restriction prevents opening a parent-folder link,
open that PDF directly from the course folder.
